# Purpose
## The primary task is to find correlations between the price of a vehcile and other attributes, such as year, make, etc. 

## The secondary task is to explore any correlation that exists between other attributes. 

# There are some preliminary preparation tasks you should complete before the EDA tasks.

### a.	Adequacy Check: Make sure that the dataset is large enough and representative of the problem you're trying to solve. Ensure it includes relevant variables for the analysis you're aiming to perform. 

The datasets contain over 9000 rows of data, more than enough to establish findings. However, if we want to predict the price of a Tesla Model Y or a Ford F-150, we only have a dozen or so rows for those specific makes/models. We will proceed with the EDA, but adjust our goals based on appropriate size of data. For example, we might target a Hyundai Palisade if this make/model contains a larger dataset than other makes/models.

### b.	Data Integrity: Check for source information and understand its data collection methodology and if there are any potential biases in the dataset. Sometimes datasets come with unclear or misleading variable names, so it’s essential to interpret them in context. 

The data comes from https://www.kaggle.com/datasets/ahmedgmy/cars-raw-dataset

Kaggle is a common dataset sharing website and while I do not know the author Ahmed Gamal personally, I verified the data by searching the VINs of cars within the dataset on CARFAX. CARFAX verified several makes/models of different cars when I searched the VINs, verifiying the authenticity of the dataset. 

There are various scores to reliability, dealer seller rating, etc. While these could be skewed or bias, we will press forward with our analysis. However the data seems to be consistent with normal Google/seller reviews. For example, BG Motorcars in IL had a Seller Rating of 4.1 and on Google, it had 3.9 out of 5 stars. 

1.	Describe the Variable Characteristics with Summary Statistics  
a.	Provide description of the variables you are using in this project.  

In [ ]:
import pandas as pd
from prettytable import PrettyTable


notes = {
    'Price':'Target variable we want to predict',
    'SellerType':'Either private or dealer',
    'DealType':'Based on MSRP/KBB Value of car',
    'VIN':'Used for identification',
    'Stock#':"Used for identification"
}

# Load the data
cars_df = pd.read_csv("cars_raw.csv")

# Create the PrettyTable
table = PrettyTable()
table.field_names = ["Column Name", "Data Type", "Example Data", "Notes"]

# Helper function to classify type
def classify_type(series):
    if pd.api.types.is_numeric_dtype(series):
        return "Numerical"
    elif series.nunique(dropna=True) == 2:
        return "Binary"
    else:
        return "Categorical"

# Fill the table
for col in cars_df.columns:
    col_name = col
    data_type = classify_type(cars_df[col])
    example_data = str(cars_df[col].iloc[0])
    notes = ""
    table.add_row([col_name, data_type, example_data, notes])

# Print the table
print(table)


+-----------------------+-------------+------------------------------------------------------------------+-------+
|      Column Name      |  Data Type  |                           Example Data                           | Notes |
+-----------------------+-------------+------------------------------------------------------------------+-------+
|          Year         |  Numerical  |                               2019                               |       |
|          Make         | Categorical |                              Toyota                              |       |
|         Model         | Categorical |                            Sienna SE                             |       |
|        Used/New       | Categorical |                               Used                               |       |
|         Price         | Categorical |                             $39,998                              |       |
|     ConsumerRating    |  Numerical  |                               4.6       

2.	Data Cleaning and Preparation 

a.	Handle missing values or outliers  

In [17]:
# Count of missing values per column
missing_sum = cars_df.isnull().sum()

# Percentage of missing values per column
missing_mean = cars_df.isnull().mean()

# Rows with any missing values
missing_rows = cars_df[cars_df.isnull().any(axis=1)]

# Print summary
print(f"Total missing values in DataFrame: {missing_sum.sum()}")
print(f"Average missing percentage per column: {missing_mean.mean():.2%}")
# While we could use some imputation for things such as missing miles, or price, that would require us to find associated make/models of the same year, and dropping the frames is easier.
cleaned_cars_df = cars_df.dropna()


# Outliers will NOT be dropped but will be standardized


Total missing values in DataFrame: 222
Average missing percentage per column: 0.07%


In [ ]:
# Count unique values and sort by frequency (descending)
mode_counts = cleaned_cars_df['Model'].value_counts()

# Print the top 20 models by count
print("Top 20 unique values in 'Model' sorted by count:")
print(mode_counts.head(20))




'''
The most popular models for sale are Grand Cherokee Limited, CR-V, and X5. 
When we compare models, we have to be cognizant of variances between generations.

For example, the Grand Cherokee is fairly easy because it's the same car/trim across several rows.
At a glance, we assume the X5 is probably the same as other X5s, but the xDrive40i and xDrive35i are different generations,
leading to variances in pricing and reliability. 

Further analysis might compare models against models, but we should also save off the top 3 models against themselves for model-specific analysis
'''

grand_cherokee_df = cleaned_cars_df[cleaned_cars_df['Model'] == "Grand Cherokee Limited"]
crv_df = cleaned_cars_df[cleaned_cars_df['Model'] == "CR-V EX-L"]
x5_drive40i = cleaned_cars_df[cleaned_cars_df['Model'] == "X5 xDrive40i"]

top_3_cars_df= [grand_cherokee_df,crv_df,x5_drive40i]

Top 20 unique values in 'Model' sorted by count:
Model
Grand Cherokee Limited    265
CR-V EX-L                 189
X5 xDrive40i              164
XT5 Luxury                140
RAV4 LE                   112
GX 460 Base               111
X3 xDrive30i              108
Pilot EX-L                102
CX-5 Grand Touring         99
1500 Laramie               87
C-Class C 300              83
Corolla LE                 79
CR-V EX                    77
QX60 Luxe                  75
RX 350 Base                72
GLC 300 Base 4MATIC        72
RAV4 XLE                   71
4Runner Limited            69
C-Class C 300 4MATIC       65
330 i xDrive               64
Name: count, dtype: int64


In [21]:
grand_cherokee_df

,Year,Make,Model,Used/New,Price,ConsumerRating,ConsumerReviews,SellerType,SellerName,SellerRating,...,InteriorColor,Drivetrain,MinMPG,MaxMPG,FuelType,Transmission,Engine,VIN,Stock#,Mileage
243,2019,Jeep,Grand Cherokee Limited,Used,"$35,500",4.8,224,Dealer,Scott Clark Toyota,4.7,...,Black,Four-wheel Drive,18,25,Gasoline,8-Speed Automatic,3.6L V6 24V MPFI DOHC,1C4RJFBG0KC555534,SH55534,32858
276,2017,Jeep,Grand Cherokee Limited,Used,"$27,575",4.8,334,Dealer,Bill Luke Chrysler Jeep Dodge RAM,4.3,...,Black,Rear-wheel Drive,19,26,Gasoline,8-Speed Automatic,3.6L V6 24V MPFI DOHC,1C4RJEBG2HC888367,CU55374X,16929
300,2019,Jeep,Grand Cherokee Limited,Used,"$34,991",4.8,224,Dealer,Mercedes-Benz of Houston North,4.5,...,–,Rear-wheel Drive,19,26,Gasoline,8-Speed Automatic,3.6L V6 24V MPFI DOHC,1C4RJEBG4KC530551,KC530551,33995
372,2019,Jeep,Grand Cherokee Limited,Used,"$39,439",4.8,224,Dealer,EchoPark Automotive Salt Lake City Delivery Ce...,3.0,...,Black,Four-wheel Drive,18,25,Gasoline,8-Speed Automatic,3.6L V6 24V MPFI DOHC,1C4RJFBG4KC635712,PKC635712,39412
400,2018,Jeep,Grand Cherokee Limited,Used,"$33,663",4.8,395,Dealer,Brickner's of Antigo Inc.,4.3,...,Black,Four-wheel Drive,18,25,Gasoline,8-Speed Automatic,3.6L V6 24V MPFI DOHC,1C4RJFBG4JC484286,4805,43807
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9133,2018,Jeep,Grand Cherokee Limited,Used,"$33,998",4.8,395,Dealer,Walser Subaru,4.4,...,Black,Four-wheel Drive,18,25,Gasoline,Automatic,Regular Unleaded V-6 3.6 L/220,1C4RJFBG0JC436851,14BX228P,30923
9224,2019,Jeep,Grand Cherokee Limited,Used,"$29,900",4.8,224,Dealer,"Napleton Hyundai, Genesis of Hazelwood",4.8,...,Black,Four-wheel Drive,18,25,Gasoline,8-Speed Automatic,3.6L V6 24V MPFI DOHC,1C4RJFBG4KC804031,RT2955A,60434
9261,2018,Jeep,Grand Cherokee Limited,Used,"$32,990",4.8,395,Dealer,Susquehanna Chrysler Dodge Jeep RAM,4.8,...,Black,Four-wheel Drive,18,25,Gasoline,8-Speed Automatic,3.6L V6 24V MPFI DOHC,1C4RJFBG7JC245203,2410B,33639
9269,2019,Jeep,Grand Cherokee Limited,Used,"$40,971",4.8,224,Dealer,Hertrich Chrysler Dodge Jeep Ram of Elkton,4.6,...,Black,Four-wheel Drive,18,25,Gasoline,8-Speed Automatic,3.6L V6 24V MPFI DOHC,1C4RJFBG5KC847082,R30107,42579


grand_cherokee_df

Conclusions:

Manual research verified the authenticity of data. By comparing VINs on Carfax with the VINs from the dataset, we verified the make/model/year of various cars.

Some of the dealers with low ratings, such as CarLotz Denver and EchoPark Automotive Salt Lake City Delivery Center, were permanently closed. 